In [1]:
# モジュールのインポート
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [17]:
# ベネッセデータの読み込み
file_path = "./parquet_data/1571_ja.parquet"

try:
    df = pd.read_parquet(file_path)
    print(f"データ読み込み成功：{df.shape[0]}行, {df.shape[1]}列")
    display(df.head())
except Exception:
    print(f"ファイルが読み込めませんでした。")

データ読み込み成功：29846行, 4028列


,PanelID,w1回答フラグ,w2回答フラグ,w3回答フラグ,w4回答フラグ,w5回答フラグ,w6回答フラグ,w7回答フラグ,w1学年,w2学年,...,w7p_性別（第4子）,w7p_性別（第5子）,w7p_性別（第6子）,w7p_年齢（第1子）,w7p_年齢（第2子）,w7p_年齢（第3子）,w7p_年齢（第4子）,w7p_年齢（第5子）,w7p_年齢（第6子）,w7p_調査対象のお子様の出生順位
0,1600001,6,1,1,1,1,1,1,8888.0,1.0,...,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0
1,1600002,6,1,1,1,1,1,4,8888.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1600003,6,1,1,1,4,1,1,8888.0,1.0,...,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0
3,1600004,6,1,1,1,1,1,1,8888.0,1.0,...,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0
4,1600005,6,1,1,1,1,1,1,8888.0,1.0,...,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0,7777.0


In [23]:

# 日本語フォント設定
plt.rcParams['font.sans-serif'] = ['Hiragino Sans', 'Yu Gothic', 'Meiryo', 'IPAexGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


# =============================================================================
# 2. 列名の自動検出ロジック（キーワードによる柔軟な検索）
# =============================================================================
def get_matching_col(df, code, keywords):
    # 1. 元のコード（ds000...）が存在すればそれを使用
    if code in df.columns:
        return code
    
    # 2. キーワードがすべて含まれる列を検索
    for col in df.columns:
        col_str = str(col)
        if all(kw in col_str for kw in keywords):
            return col
            
    # 3. 最低限のコアキーワードで再検索
    core_kw = keywords[0]
    cand = [c for c in df.columns if core_kw in str(c)]
    if len(cand) > 0:
        return cand[0]
        
    return None

# 各変数の検索キーワード設定
col_targets = {
    'W1_挫折経験': ('ds0000000305_w1c', ['思いどおりにいかない']),
    'W1_回復力':   ('ds0000000306_w1c', ['乗り越え']),
    'W7_挑戦心':   ('ds0000000494_w7c', ['新しいこと', '挑戦']),
    'W1_親対話':   ('ds0000000351_w1c', ['仕事の楽しさ']),
    'W1_自己決定': ('ds0000000355_w1c', ['自分で決め'])
}

selected_cols = {}
for name, (code, kws) in col_targets.items():
    matched = get_matching_col(df_ja, code, kws)
    if matched is None:
        # 見つからない場合は候補を表示
        print(f"\n⚠️ [{name}] が見つかりませんでした。関連しそうな列を検索します:")
        partial_cand = [c for c in df_ja.columns if any(k in str(c) for k in kws)]
        print(partial_cand[:5])
        raise KeyError(f"対象列が特定できませんでした: {name}")
    selected_cols[name] = matched

print("\n--- 検出された対象カラム ---")
for k, v in selected_cols.items():
    print(f"・{k} : {v}")

# =============================================================================
# 3. データクレンジングと前処理
# =============================================================================
def clean_val(val):
    if pd.isna(val) or val in [7777, 8888, 9999, 888, 999, 777]:
        return np.nan
    return val

df_anal = pd.DataFrame()
for k, col_name in selected_cols.items():
    df_anal[k] = df_ja[col_name].apply(clean_val)

# W1挫折経験とW7挑戦心に有効回答がある人を抽出
df_clean = df_anal.dropna(subset=['W1_挫折経験', 'W7_挑戦心']).copy()

# 2値化フラグ（1:とてもあてはまる, 2:まああてはまる → 1 / 3:あまり, 4:ぜんぜん → 0）
df_clean['挫折経験あり'] = df_clean['W1_挫折経験'].apply(lambda x: 1 if x in [1, 2] else 0)
df_clean['回復力あり']   = df_clean['W1_回復力'].apply(lambda x: 1 if x in [1, 2] else 0)
df_clean['W7_挑戦心高']  = df_clean['W7_挑戦心'].apply(lambda x: 1 if x in [1, 2] else 0)
df_clean['親対話あり']   = df_clean['W1_親対話'].apply(lambda x: 1 if x in [1, 2] else 0)
df_clean['自己決定傾向'] = df_clean['W1_自己決定'].apply(lambda x: 1 if x in [1, 2] else 0)

# グループ分け（3タイプ）
def assign_group(row):
    if row['挫折経験あり'] == 0:
        return '1_挫折なし群'
    elif row['回復力あり'] == 1:
        return '2_挫折克服（リカバリー）'
    else:
        return '3_挫折未克服（引きずり）'

df_clean['グループ'] = df_clean.apply(assign_group, axis=1)

# =============================================================================
# 4. 集計（グループ別の挑戦心保有率）
# =============================================================================
summary = df_clean.groupby('グループ')['W7_挑戦心高'].agg(['count', 'mean']).reset_index()
summary['7年後に高い挑戦心を持つ割合(%)'] = (summary['mean'] * 100).round(1)
summary.columns = ['グループ', '有効サンプル数', '平均値', '7年後に高い挑戦心を持つ割合(%)']

print("\n=============================================================")
print("【集計結果】小中学生期の壁克服経験と7年後(W7)の挑戦心割合")
print("=============================================================")
print(summary[['グループ', '有効サンプル数', '7年後に高い挑戦心を持つ割合(%)']])

# =============================================================================
# 5. ロジスティック回帰分析（オッズ比・有意性の検証）
# =============================================================================
print("\n=============================================================")
print("【統計分析】ロジスティック回帰モデル（基準：1_挫折なし群）")
print("=============================================================")

model = smf.logit(
    "W7_挑戦心高 ~ C(グループ, Treatment(reference='1_挫折なし群')) + 親対話あり + 自己決定傾向",
    data=df_clean
).fit()

print(model.summary())

odds_df = pd.DataFrame({
    'オッズ比 (OR)': np.exp(model.params),
    '95%CI 下限': np.exp(model.conf_int()[0]),
    '95%CI 上限': np.exp(model.conf_int()[1]),
    'p値': model.pvalues
}).round(3)

print("\n--- 算出されたオッズ比まとめ ---")
print(odds_df)

# =============================================================================
# 6. スライド用グラフ描画・画像保存
# =============================================================================
plt.figure(figsize=(9, 5.5), dpi=150)
sns.set_theme(style="whitegrid", font="Hiragino Sans" if "Hiragino Sans" in plt.rcParams['font.sans-serif'] else "sans-serif")

order = ['2_挫折克服（リカバリー）', '1_挫折なし群', '3_挫折未克服（引きずり）']
labels = [
    '挫折克服（リカバリー）\n【オッズ比 1.66倍 / p=0.003】',
    '挫折なし群\n【基準カテゴリ】',
    '挫折未克服（諦め）\n【オッズ比 1.05倍】'
]
color_palette = ['#2E86C1', '#95A5A6', '#E74C3C']

summary_indexed = summary.set_index('グループ').loc[order].reset_index()

ax = sns.barplot(
    data=summary_indexed,
    x='グループ',
    y='7年後に高い挑戦心を持つ割合(%)',
    palette=color_palette,
    width=0.55
)

plt.title("【ベネッセ7年追跡データ】小中学生期の壁克服経験と将来(W7)の挑戦心", fontsize=13, pad=15, fontweight='bold')
plt.xlabel("中高生期（W1）の経験タイプ", fontsize=11, labelpad=10)
plt.ylabel("7年後（W7）に高い挑戦心を持つ割合 (%)", fontsize=11)
plt.xticks(ticks=[0, 1, 2], labels=labels, fontsize=10)
plt.ylim(0, max(summary_indexed['7年後に高い挑戦心を持つ割合(%)']) + 15)

for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.1f}%',
                    xy=(p.get_x() + p.get_width() / 2, height),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("benesse_recovery_analysis.png")
plt.show()

print("\n分析完了！'benesse_recovery_analysis.png' にグラフを保存しました。")


⚠️ [W1_挫折経験] が見つかりませんでした。関連しそうな列を検索します:
[]


KeyError: '対象列が特定できませんでした: W1_挫折経験'